 # Imports
 

In [13]:
import sys
import os
sys.path.append(os.path.join(os.getcwd(), '..', 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')


from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error

import tensorflow as tf
from tensorflow.keras.models import Sequential 
from tensorflow.keras.layers import LSTM , Dense , Dropout
from tensorflow.keras.callbacks import EarlyStopping

from database import load_zone_data

print('ALl LIBRARIES IMPORTED SUCCESSFULLY')

ALl LIBRARIES IMPORTED SUCCESSFULLY


In [14]:
# creating sequences for LSTM
def create_sequences(data, window_size = 14):
    X , y = [] , []
    for i in range(window_size , len(data)):
        X.append(data[i-window_size:i])
        y.append(data[i])
    
    return np.array(X), np.array(y)


In [ ]:
#  load zone and build sequences
zone = 'zone2_coastal_west'
zone_df = load_zone_data(zone)

VARIABLES = ['temperature', 'precipitation', 'windspeed', 'humidity']
WINDOW    = 14

all_X = []
all_y = []

scalers = {}  # one scaler per city — remember to inverse transform later

for city in zone_df['city'].unique():
    
    # get city data
    city_df = zone_df[zone_df['city'] == city].copy()
    city_df = city_df.set_index('date')
    city_df = city_df.asfreq('D')
    city_df = city_df.fillna(method='ffill')
    
    # keep only 4 weather variables
    data = city_df[VARIABLES].values
    
    # scale to 0-1
    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(data)
    scalers[city] = scaler
    
    # create sequences
    X, y = create_sequences(data_scaled, WINDOW)
    
    all_X.append(X)
    all_y.append(y)

# stack all cities
X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)

print(f"X shape: {X_all.shape}")
print(f"y shape: {y_all.shape}")
print(f"Expected X: (samples, 14, 4)")

X shape: (53000, 14, 4)
y shape: (53000, 4)
Expected X: (samples, 14, 4)


For each city in zone:

    1. Filter that city's rows
    2. Set date as index, fill gaps
    3. Take only 4 weather columns → numpy array
    4. Scale to 0-1 using MinMaxScaler
    5. Save scaler for later inverse transform
    6. Create sliding windows → X and y
    7. Append to lists

Stack all cities → one big X_all and y_all

In [16]:
# train-test split
train_size  = int(len(X_all) * 0.80)

X_train = X_all[:train_size]
X_test  = X_all[train_size:]
y_train = y_all[:train_size]
y_test  = y_all[train_size:]

print(f'X_train : {X_train.shape}')
print(f'X_test : {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'y_test : {y_test.shape}')

X_train : (42400, 14, 4)
X_test : (10600, 14, 4)
y_train: (42400, 4)
y_test : (10600, 4)


In [ ]:
model = Sequential([
    
    # first LSTM
    LSTM(64, return_sequences=True, input_shape=(14, 4)),
    Dropout(0.2),

    LSTM(32, return_sequences=False),
    Dropout(0.2),

    Dense(4)
])

model.compile(optimizer='adam', loss='mse')
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 14, 64)         │        17,664 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 14, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 4)              │           132 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 30,212 (118.02 KB)

 Trainable params: 30,212 (118.02 KB)

 Non-trainable params: 0 (0.00 B)

In [18]:
#  — Train LSTM
early_stop = EarlyStopping(
    monitor   = 'val_loss',
    patience  = 10,
    restore_best_weights = True
)

history = model.fit(
    X_train, y_train,
    epochs          = 100,
    batch_size      = 32,
    validation_split = 0.1,
    callbacks       = [early_stop],
    verbose         = 1
)

Epoch 1/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 11s 7ms/step - loss: 0.0110 - val_loss: 0.0067
Epoch 2/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 8s 7ms/step - loss: 0.0063 - val_loss: 0.0053
Epoch 3/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0051 - val_loss: 0.0047
Epoch 4/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 11s 9ms/step - loss: 0.0045 - val_loss: 0.0046
Epoch 5/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 14s 11ms/step - loss: 0.0042 - val_loss: 0.0044
Epoch 6/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0041 - val_loss: 0.0044
Epoch 7/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0041 - val_loss: 0.0044
Epoch 8/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0040 - val_loss: 0.0045
Epoch 9/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - loss: 0.0040 - val_loss: 0.0044
Epoch 10/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 9s 8ms/step - loss: 0.0040 - val_loss: 0.0043
Epoch 11/100
1193/1193 ━━━━━━━━━━━━━━━━━━━━ 10s 8ms/step - loss: 0.0040 - val_loss: 0.0044
Epoch 12/10

In [ ]:
# Evaluate LSTM
y_pred_scaled = model.predict(X_test)

first_city = list(scalers.keys())[0]
scaler = scalers[first_city]

y_pred_real = scaler.inverse_transform(y_pred_scaled)
y_test_real = scaler.inverse_transform(y_test)

lstm_rmse = {}
print("LSTM — zone2_coastal_west")
print("-" * 45)
for i, variable in enumerate(['temperature', 'precipitation',
                               'windspeed', 'humidity']):
    rmse = np.sqrt(mean_squared_error(
        y_test_real[:, i], y_pred_real[:, i]))
    mae  = mean_absolute_error(
        y_test_real[:, i], y_pred_real[:, i])
    print(f"{variable:<15} RMSE: {rmse:.3f}   MAE: {mae:.3f}")
    lstm_rmse[variable] = rmse

332/332 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step
LSTM — zone2_coastal_west
---------------------------------------------
temperature     RMSE: 0.370   MAE: 0.271
precipitation   RMSE: 9.045   MAE: 3.742
windspeed       RMSE: 4.181   MAE: 3.012
humidity        RMSE: 5.020   MAE: 3.329


In [ ]:
# Final comparison all models
rf_rmse   = {'temperature': 0.676, 'precipitation': 9.870,
             'windspeed': 2.703,   'humidity': 6.402}

xgb_rmse  = {'temperature': 0.711, 'precipitation': 10.378,
             'windspeed': 2.793,   'humidity': 6.568}

lgbm_rmse = {'temperature': 0.661, 'precipitation': 9.530,
             'windspeed': 2.582,   'humidity': 6.202}

print("=" * 65)
print(f"{'Final Model Comparison — zone2_coastal_west':^65}")
print("=" * 65)
print(f"{'Variable':<15} {'VAR':>7} {'RF':>7} {'XGB':>7} "
      f"{'LGBM':>7} {'LSTM':>7} {'Winner':>10}")
print("-" * 65)

var_rmse  = {'temperature': 2.013, 'precipitation': 16.737,
             'windspeed': 5.159,   'humidity': 11.377}

all_scores = {
    'VAR':      var_rmse,
    'RF':       rf_rmse,
    'XGBoost':  xgb_rmse,
    'LightGBM': lgbm_rmse,
    'LSTM':     lstm_rmse
}

final_winners = {}

for variable in ['temperature', 'precipitation',
                 'windspeed', 'humidity']:
    scores = {m: all_scores[m][variable] for m in all_scores}
    winner = min(scores, key=lambda x: scores[x])
    final_winners[variable] = winner

    print(f"{variable:<15} "
          f"{var_rmse[variable]:>7.3f} "
          f"{rf_rmse[variable]:>7.3f} "
          f"{xgb_rmse[variable]:>7.3f} "
          f"{lgbm_rmse[variable]:>7.3f} "
          f"{lstm_rmse[variable]:>7.3f} "
          f"{winner:>10}")

print("=" * 65)
print("\nFinal winners:")
for var, win in final_winners.items():
    print(f"  {var:<15} → {win}")

           Final Model Comparison — zone2_coastal_west           
Variable            VAR      RF     XGB    LGBM    LSTM     Winner
-----------------------------------------------------------------
temperature       2.013   0.676   0.711   0.661   0.370       LSTM
precipitation    16.737   9.870  10.378   9.530   9.045       LSTM
windspeed         5.159   2.703   2.793   2.582   4.181   LightGBM
humidity         11.377   6.402   6.568   6.202   5.020       LSTM

Final winners:
  temperature     → LSTM
  precipitation   → LSTM
  windspeed       → LightGBM
  humidity        → LSTM


In [23]:
# Saving LSTM Model

import os

save_dir = '../saved_models/zone2_coastal_west'
os.makedirs(save_dir, exist_ok=True)

# save LSTM model in keras format
model.save(f'{save_dir}/lstm_model.keras')

# save scalers — needed for inverse transform at prediction time
import joblib
joblib.dump(scalers, f'{save_dir}/scalers.pkl')

print(f"LSTM model saved → {save_dir}/lstm_model.keras")
print(f"Scalers saved    → {save_dir}/scalers.pkl")

LSTM model saved → ../saved_models/zone2_coastal_west/lstm_model.keras
Scalers saved    → ../saved_models/zone2_coastal_west/scalers.pkl


## DL Model (LSTM) — Key Findings (zone2_coastal_west)

### Final Results
| Variable      | VAR    | RF    | XGB    | LightGBM | LSTM  | Winner   |
|---------------|--------|-------|--------|----------|-------|----------|
| Temperature   | 2.013  | 0.676 | 0.711  | 0.661    | 0.370 | LSTM     |
| Precipitation | 16.737 | 9.870 | 10.378 | 9.530    | 9.045 | LSTM     |
| Windspeed     | 5.159  | 2.703 | 2.793  | 2.582    | 4.181 | LightGBM |
| Humidity      | 11.377 | 6.402 | 6.568  | 6.202    | 5.020 | LSTM     |

### Key Findings

**1. LSTM wins on 3 out of 4 variables**
Temperature improved 82% over VAR — from 2.013 to 0.370.
Long-term memory captures seasonal patterns RF/XGB cannot.

**2. LightGBM wins on windspeed**
Windspeed is chaotic — less seasonal structure.
Tree models handle chaotic variables better than sequence models.
This validates the zone-based model selection approach.

**3. Progressive improvement across model families**
VAR → RF → LightGBM → LSTM shows clear progression.
Each family captures patterns the previous one missed.

**4. Decision for production pipeline**
LSTM     → temperature, precipitation, humidity
LightGBM → windspeed
Both saved to saved_models/zone2_coastal_west/

### Training observations
- Converged at epoch 29 out of 100
- EarlyStopping prevented overfitting
- train_loss ≈ val_loss → no overfitting detected
- CPU training time → ~6 minutes for 42400 samples